# SeatGuard — Station 4: Cosmetic Defect Detection

Trains YOLOv8 for end-of-line cosmetic defects on the completed seat.

**Dataset:** Fabric Defect Detection from Kaggle (~10,000 images)  
Defect classes remapped to Station 4 cosmetic categories:

| Source class | Station 4 class |
|---|---|
| hole, stain, thread | `wrinkle_puckering` |
| broken, cut, scratch | `foam_showthrough` |

**Expected output:** `cosmetics.onnx` for `models/station_4/`

**Runtime:** GPU → Runtime → Change runtime type → T4 GPU

## Step 1 — Install

In [ ]:
!pip install ultralytics kaggle --quiet
import torch
print(torch.__version__, torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Step 2 — Upload kaggle.json and download dataset

Get `kaggle.json` from [kaggle.com](https://kaggle.com) → Settings → API → Create New Token

In [ ]:
from google.colab import files
import os

print('Upload kaggle.json:')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# Download fabric defect detection dataset (~10k images, annotated)
!kaggle datasets download -d nexuswho/fabric-defect-detection -p /content/raw --unzip

# Check structure
for root, dirs, flist in os.walk('/content/raw'):
    level = root.replace('/content/raw', '').count(os.sep)
    if level <= 2:
        print('  '*level + os.path.basename(root) + f'/ ({len(flist)} files)')

## Step 3 — Check annotation format

In [ ]:
import os, glob

# Find what annotation files exist
xmls  = glob.glob('/content/raw/**/*.xml',  recursive=True)[:3]
txts  = glob.glob('/content/raw/**/*.txt',  recursive=True)[:3]
jsons = glob.glob('/content/raw/**/*.json', recursive=True)[:3]
imgs  = glob.glob('/content/raw/**/*.jpg',  recursive=True)[:3] + \
        glob.glob('/content/raw/**/*.png',  recursive=True)[:3]

print(f'XML  files: {len(glob.glob("/content/raw/**/*.xml",  recursive=True))}')
print(f'TXT  files: {len(glob.glob("/content/raw/**/*.txt",  recursive=True))}')
print(f'JSON files: {len(glob.glob("/content/raw/**/*.json", recursive=True))}')
print(f'JPG  files: {len(glob.glob("/content/raw/**/*.jpg",  recursive=True))}')
print(f'PNG  files: {len(glob.glob("/content/raw/**/*.png",  recursive=True))}')

# Peek at a label file
if txts:
    print('\nSample TXT label:')
    with open(txts[0]) as f: print(f.read()[:300])
elif xmls:
    print('\nSample XML label:')
    with open(xmls[0]) as f: print(f.read()[:500])
elif jsons:
    print('\nSample JSON label (first 300 chars):')
    with open(jsons[0]) as f: print(f.read()[:300])

## Step 4 — Convert and remap classes to SeatGuard Station 4 format

Remaps all fabric defect classes to 2 Station 4 cosmetic classes:
- `0: wrinkle_puckering` — surface shape defects (hole, stain, uneven)
- `1: foam_showthrough`  — material integrity defects (cut, tear, scratch)

In [ ]:
import os, shutil, random, yaml, glob
import xml.etree.ElementTree as ET
from pathlib import Path

# ── Remap: source class name → Station 4 class ───────────────────────────────
# Adjust LEFT side based on what class names appear in the dataset labels
CLASS_MAP = {
    # shape/surface defects → wrinkle_puckering (class 0)
    'hole':          0,
    'stain':         0,
    'thread':        0,
    'weaving':       0,
    'weft':          0,
    'warp':          0,
    'netting':       0,
    'pilling':       0,
    'wrinkle':       0,
    'fold':          0,
    # material integrity defects → foam_showthrough (class 1)
    'broken':        1,
    'cut':           1,
    'scratch':       1,
    'tear':          1,
    'crack':         1,
    'missing':       1,
    'defect':        1,
    'damage':        1,
}
CLASS_NAMES = ['wrinkle_puckering', 'foam_showthrough']
# ─────────────────────────────────────────────────────────────────────────────

OUT = Path('/content/station4_dataset')
RAW = Path('/content/raw')

# Detect annotation format
all_xmls  = list(RAW.rglob('*.xml'))
all_txts  = list(RAW.rglob('*.txt'))
all_imgs  = list(RAW.rglob('*.jpg')) + list(RAW.rglob('*.png'))

print(f'Images: {len(all_imgs)}  XMLs: {len(all_xmls)}  TXTs: {len(all_txts)}')

def voc_to_yolo(xmin, ymin, xmax, ymax, w, h):
    cx = ((xmin+xmax)/2)/w; cy = ((ymin+ymax)/2)/h
    bw = (xmax-xmin)/w;     bh = (ymax-ymin)/h
    return cx, cy, bw, bh

def parse_xml(xml_path):
    tree = ET.parse(xml_path); root = tree.getroot()
    size = root.find('size')
    W = int(size.find('width').text); H = int(size.find('height').text)
    boxes = []
    for obj in root.findall('object'):
        name = obj.find('name').text.strip().lower()
        b = obj.find('bndbox')
        xmin,ymin,xmax,ymax = int(float(b.find('xmin').text)), int(float(b.find('ymin').text)), int(float(b.find('xmax').text)), int(float(b.find('ymax').text))
        boxes.append((name, xmin, ymin, xmax, ymax))
    return W, H, boxes

# Build image→xml map
xml_by_stem = {p.stem: p for p in all_xmls}

# If YOLO txt labels already exist, check if classes.txt defines names
classes_txt = list(RAW.rglob('classes.txt'))
orig_classes = []
if classes_txt:
    with open(classes_txt[0]) as f:
        orig_classes = [l.strip().lower() for l in f.readlines() if l.strip()]
    print(f'Original classes from classes.txt: {orig_classes}')

# Split images
paired = [p for p in all_imgs if p.stem in xml_by_stem] if all_xmls else all_imgs
random.seed(42); random.shuffle(paired)
n = len(paired)
splits = {'train': paired[:int(n*0.8)], 'valid': paired[int(n*0.8):int(n*0.9)], 'test': paired[int(n*0.9):]}
print(f'Total paired: {n} → train:{len(splits["train"])} valid:{len(splits["valid"])} test:{len(splits["test"])}')

converted = skipped = 0
found_classes = set()

for split, imgs in splits.items():
    (OUT/split/'images').mkdir(parents=True, exist_ok=True)
    (OUT/split/'labels').mkdir(parents=True, exist_ok=True)

    for img_path in imgs:
        lines = []

        if all_xmls and img_path.stem in xml_by_stem:
            # VOC XML path
            try:
                W, H, boxes = parse_xml(xml_by_stem[img_path.stem])
            except Exception:
                skipped += 1; continue
            for (name, xmin, ymin, xmax, ymax) in boxes:
                found_classes.add(name)
                # Try exact match, then partial match
                cls_id = CLASS_MAP.get(name)
                if cls_id is None:
                    for key in CLASS_MAP:
                        if key in name or name in key:
                            cls_id = CLASS_MAP[key]; break
                if cls_id is None:
                    cls_id = 1  # fallback: foam_showthrough
                cx, cy, bw, bh = voc_to_yolo(xmin, ymin, xmax, ymax, W, H)
                lines.append(f'{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

        elif all_txts:
            # Already YOLO format — remap class IDs
            txt_path = img_path.parent / (img_path.stem + '.txt')
            if not txt_path.exists():
                # Try labels subdir
                txt_path = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
            if txt_path.exists():
                with open(txt_path) as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) < 5: continue
                        orig_id = int(parts[0])
                        if orig_classes and orig_id < len(orig_classes):
                            name = orig_classes[orig_id]
                            found_classes.add(name)
                            cls_id = CLASS_MAP.get(name, 1)
                        else:
                            cls_id = orig_id % 2  # fallback
                        lines.append(f'{cls_id} ' + ' '.join(parts[1:]))

        shutil.copy(img_path, OUT/split/'images'/img_path.name)
        (OUT/split/'labels'/(img_path.stem+'.txt')).write_text('\n'.join(lines))
        converted += 1

print(f'Converted: {converted}  Skipped: {skipped}')
print(f'Found source classes: {found_classes}')

# Write data.yaml
data_yaml_path = str(OUT/'data.yaml')
cfg = {'path': str(OUT), 'train': 'train/images', 'val': 'valid/images',
       'test': 'test/images', 'nc': 2, 'names': CLASS_NAMES}
with open(data_yaml_path, 'w') as f: yaml.dump(cfg, f)
print('data.yaml:', cfg)

## Step 5 — Train YOLOv8

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    lr0=0.01,
    patience=10,
    augment=True,
    degrees=15.0,
    fliplr=0.5,
    flipud=0.3,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    mosaic=1.0,
    project='seatguard_station4',
    name='cosmetics',
    exist_ok=True,
    device=0,
    conf=0.45,
    verbose=True,
)
print('Done:', results.save_dir)

## Step 6 — Evaluate

In [ ]:
import glob
best_pt = glob.glob('**/seatguard_station4/cosmetics*/weights/best.pt', recursive=True)
if not best_pt:
    best_pt = glob.glob('**/best.pt', recursive=True)
best_pt = best_pt[0]
print('Best weights:', best_pt)

trained_model = YOLO(best_pt)
metrics = trained_model.val(data=data_yaml_path, conf=0.45)

print(f'\nmAP50      : {metrics.box.map50:.4f}')
print(f'mAP50-95   : {metrics.box.map:.4f}')
print(f'Precision  : {metrics.box.mp:.4f}')
print(f'Recall     : {metrics.box.mr:.4f}')
print('\nPer-class AP50:')
for cls_name, ap in zip(trained_model.names.values(), metrics.box.ap50):
    print(f'  {cls_name:25s}: {ap:.4f}')

## Step 7 — Export to ONNX

In [ ]:
import onnx, os

trained_model.export(format='onnx', imgsz=640, opset=17, dynamic=True, simplify=True)
onnx_path = best_pt.replace('.pt', '.onnx')
print(f'ONNX: {onnx_path}')

m = onnx.load(onnx_path)
onnx.checker.check_model(m)
print('ONNX check passed')
print(f'Size: {os.path.getsize(onnx_path)/1e6:.1f} MB')

## Step 8 — Download

In [ ]:
import shutil, os
from google.colab import files

shutil.copy(onnx_path, 'cosmetics.onnx')
print(f'Size: {os.path.getsize("cosmetics.onnx")/1e6:.1f} MB')
files.download('cosmetics.onnx')
print('Place at: models/station_4/cosmetics.onnx')

## After downloading

Update `configs/stations/station_4_complete.yaml`:
```yaml
model:
  backend: "onnx"
  model_type: "yolo_cosmetic_defects"
  weights_path: "models/station_4/cosmetics.onnx"
  version: "v1.0.0"
  threshold: 0.45
  class_names:
    - "wrinkle_puckering"
    - "foam_showthrough"
```

## If Step 2 dataset download fails
Try this alternative on Kaggle:
```python
!kaggle datasets download -d rmasters1/fabric-defect-yolo -p /content/raw --unzip
```
That dataset is already in YOLO format — skip the XML conversion in Step 4.